# Demo: Synonym Table Schema

Demonstrates creating a synonym table and rows using the fixed schema defined in `scripts/APIs_pipe/schema.py`.

Run from the project root.

In [2]:
import pandas as pd

from scripts.APIs_pipe.schema import (
    SYNONYM_COLUMNS,
    UNAVAILABLE,
    empty_synonym_table,
    make_synonym_row,
)

## 1. Inspect the schema

In [3]:
print("Schema columns:")
for col in SYNONYM_COLUMNS:
    print(f"  {col}")

Schema columns:
  api_name
  kingdom
  phylum
  class
  family
  subfamily
  genus
  species
  author
  publication_name
  publication_year
  status
  source_name
  api_link
  api_internal_id


## 2. Create an empty table

In [4]:
table = empty_synonym_table()
print(f"Empty table shape: {table.shape}")
table

Empty table shape: (0, 15)


,api_name,kingdom,phylum,class,family,subfamily,genus,species,author,publication_name,publication_year,status,source_name,api_link,api_internal_id


## 3. Create a fully-specified row

In [6]:
row_full = make_synonym_row(
    api_name="GBIF",
    source_name="Encycl. Méth. Bot. 1(1): 111",
    kingdom="Fungi",
    phylum="Basidiomycota",
    **{"class": "Agaricomycetes"},
    family="Amanitaceae",
    subfamily=UNAVAILABLE,
    genus="Amanita",
    species="muscaria",
    api_internal_id="8168319",
    author="(L.) Lam.",
    publication_name="Encycl. Méth. Bot. 1(1): 111",
    publication_year="1783",
    api_link="https://www.gbif.org/species/8168319",
    status="Accepted",
)
row_full

{'api_name': 'GBIF',
 'kingdom': 'Fungi',
 'phylum': 'Basidiomycota',
 'class': 'Agaricomycetes',
 'family': 'Amanitaceae',
 'subfamily': 'U',
 'genus': 'Amanita',
 'species': 'muscaria',
 'author': '(L.) Lam.',
 'publication_name': 'Encycl. Méth. Bot. 1(1): 111',
 'publication_year': '1783',
 'status': 'Accepted',
 'source_name': 'Encycl. Méth. Bot. 1(1): 111',
 'api_link': 'https://www.gbif.org/species/8168319',
 'api_internal_id': '8168319'}

## 4. Create a partial row (missing fields default to `"U"`)

In [7]:
row_partial = make_synonym_row(
    api_name="mushroomobs",
    genus="Amanita",
    species="muscaria",
    api_internal_id="16015",
    api_link="https://mushroomobserver.org/name/show_name/16015",
    status=UNAVAILABLE,
)
row_partial

{'api_name': 'mushroomobs',
 'kingdom': 'U',
 'phylum': 'U',
 'class': 'U',
 'family': 'U',
 'subfamily': 'U',
 'genus': 'Amanita',
 'species': 'muscaria',
 'author': 'U',
 'publication_name': 'U',
 'publication_year': 'U',
 'status': 'U',
 'source_name': 'U',
 'api_link': 'https://mushroomobserver.org/name/show_name/16015',
 'api_internal_id': '16015'}

## 5. Append both rows to the table

In [8]:
table = pd.concat(
    [table, pd.DataFrame([row_full, row_partial])],
    ignore_index=True,
)
print(f"Table shape: {table.shape}")
table

Table shape: (2, 15)


,api_name,kingdom,phylum,class,family,subfamily,genus,species,author,publication_name,publication_year,status,source_name,api_link,api_internal_id
0,GBIF,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,U,Amanita,muscaria,(L.) Lam.,Encycl. Méth. Bot. 1(1): 111,1783,Accepted,Encycl. Méth. Bot. 1(1): 111,https://www.gbif.org/species/8168319,8168319
1,mushroomobs,U,U,U,U,U,Amanita,muscaria,U,U,U,U,U,https://mushroomobserver.org/name/show_name/16015,16015


## 6. Demonstrate validation errors

In [9]:
test_cases = [
    ("missing required columns",  {"genus": "Amanita", "species": "muscaria"}),
    ("bad publication_year",       {"api_name": "GBIF", "genus": "Amanita", "species": "muscaria", "api_internal_id": "8168319", "publication_year": "83"}),
    ("bad api_link",               {"api_name": "GBIF", "genus": "Amanita", "species": "muscaria", "api_internal_id": "8168319", "api_link": "not-a-url"}),
    ("bad status",                 {"api_name": "GBIF", "genus": "Amanita", "species": "muscaria", "api_internal_id": "8168319", "status": "Unknown"}),
    ("multi-word genus",           {"api_name": "GBIF", "genus": "Amanita muscaria", "species": "muscaria", "api_internal_id": "8168319"}),
]

for label, kwargs in test_cases:
    try:
        make_synonym_row(**kwargs)
        print(f"  {label}: no error (unexpected)")
    except ValueError as e:
        print(f"  {label}: ValueError — {e}")

  missing required columns: ValueError — Missing required columns: ['api_internal_id', 'api_name']
  bad publication_year: ValueError — 'Publication Year' must be a 4-digit year string or 'U', got '83'
  bad api_link: ValueError — 'Source Link' must start with 'http://' or 'https://', or be 'U', got 'not-a-url'
  bad status: ValueError — 'Status' must be one of {'Accepted', 'Synonym', 'U'}, got 'Unknown'
  multi-word genus: ValueError — 'genus' must be a single word (no whitespace) or 'U', got 'Amanita muscaria'
